# dARK E2E - Simple Authority to Resolver Flow

This notebook follows the full happy path with direct HTTP calls: create an authority, authorize a NAAN, reserve an ARK, stage Level-1 and Level-2 metadata, wait for the worker to publish, and resolve the ARK through the resolver.


## Runtime Variables

- `ADMIN_API_BASE_URL` default: `http://localhost:8000`
- `MINTER_BASE_URL` default: `http://localhost:8001`
- `RESOLVER_BASE_URL` default: `http://localhost:8002`
- `STORE_API_BASE_URL` default: `http://localhost:8003`
- `IPFS_API_BASE_URL` default: `http://localhost:5001`
- `IPFS_CLUSTER_API_URL` default: `http://localhost:9094`
- `AUTHORITY_ID` default: autogenerated
- `NAAN` default: `12345`
- `TARGET_URL` default: `https://example.org/resources/resolver-e2e`
- `POLL_INTERVAL_SECONDS` default: `2`
- `POLL_TIMEOUT_SECONDS` default: `90`


In [ ]:
import os
import time
from pprint import pprint

import requests

ADMIN_API_BASE_URL = os.getenv('ADMIN_API_BASE_URL', 'http://localhost:8000').rstrip('/')
MINTER_BASE_URL = os.getenv('MINTER_BASE_URL', 'http://localhost:8001').rstrip('/')
RESOLVER_BASE_URL = os.getenv('RESOLVER_BASE_URL', 'http://localhost:8002').rstrip('/')
STORE_API_BASE_URL = os.getenv('STORE_API_BASE_URL', 'http://localhost:8003').rstrip('/')
IPFS_API_BASE_URL = os.getenv('IPFS_API_BASE_URL', 'http://localhost:5001').rstrip('/')
IPFS_CLUSTER_API_URL = os.getenv('IPFS_CLUSTER_API_URL', 'http://localhost:9094').rstrip('/')

ADMIN_API_V1 = f'{ADMIN_API_BASE_URL}/api/v1/admin'
MINTER_API_V1 = f'{MINTER_BASE_URL}/api/v1'
RESOLVER_API_V1 = f'{RESOLVER_BASE_URL}/api/v1'

AUTHORITY_ID = os.getenv('AUTHORITY_ID', f'resolver-e2e-{int(time.time())}')
NAAN = os.getenv('NAAN', '12345').strip()
TARGET_URL = os.getenv('TARGET_URL', 'https://example.org/resources/resolver-e2e').strip()
POLL_INTERVAL_SECONDS = float(os.getenv('POLL_INTERVAL_SECONDS', '2'))
POLL_TIMEOUT_SECONDS = int(os.getenv('POLL_TIMEOUT_SECONDS', '90'))

MINTER_HEADERS = {'X-Authority-Id': AUTHORITY_ID}

L1_METADATA = {
    'title': 'Southern Patagonia Atmospheric CO2 Measurements, 2019-2024',
    'authors': ['Maria Gonzalez', 'Tomas Rojas', 'Elena Paredes'],
    'year': 2026,
    'publisher': 'LA Referencia Data Lab',
    'resource_type': 'dataset',
    'language': 'en',
    'abstract': 'Curated dataset of hourly atmospheric CO2 observations gathered at three monitoring stations in southern Patagonia between 2019 and 2024.',
    'subjects': ['atmospheric science', 'carbon cycle', 'Patagonia'],
    'rights': 'CC-BY-4.0',
    'alternate_identifiers': [{'schema': 'doi', 'value': '10.25592/dark.demo.patagonia.co2.v1'}],
    'alternate_urls': [TARGET_URL, 'https://doi.org/10.25592/dark.demo.patagonia.co2.v1'],
}

L2_XML = '''<oai_dc:dc xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/"
           xmlns:dc="http://purl.org/dc/elements/1.1/"
           xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
           xsi:schemaLocation="http://www.openarchives.org/OAI/2.0/oai_dc/ http://www.openarchives.org/OAI/2.0/oai_dc.xsd">
  <dc:title>Southern Patagonia Atmospheric CO2 Measurements, 2019-2024</dc:title>
  <dc:creator>Maria Gonzalez</dc:creator>
  <dc:creator>Tomas Rojas</dc:creator>
  <dc:creator>Elena Paredes</dc:creator>
  <dc:subject>atmospheric science</dc:subject>
  <dc:subject>carbon cycle</dc:subject>
  <dc:subject>Patagonia</dc:subject>
  <dc:description>Curated dataset of hourly atmospheric CO2 observations gathered at three monitoring stations in southern Patagonia between 2019 and 2024.</dc:description>
  <dc:publisher>LA Referencia Data Lab</dc:publisher>
  <dc:date>2026</dc:date>
  <dc:type>Dataset</dc:type>
  <dc:format>application/xml</dc:format>
  <dc:identifier>10.25592/dark.demo.patagonia.co2.v1</dc:identifier>
  <dc:language>en</dc:language>
  <dc:rights>CC-BY-4.0</dc:rights>
</oai_dc:dc>'''.strip()

ARK = None

def show(response):
    print('status:', response.status_code)
    content_type = response.headers.get('content-type', '')
    if 'json' in content_type:
        pprint(response.json())
    else:
        print(response.text)

def wait_until_published(ark):
    deadline = time.time() + POLL_TIMEOUT_SECONDS
    last_body = None
    while time.time() < deadline:
        response = requests.get(f'{MINTER_API_V1}/arks/{ark}', headers=MINTER_HEADERS, timeout=180)
        last_body = response.json()
        print('worker poll:', last_body.get('state'), last_body.get('ark'))
        if response.status_code == 200 and last_body.get('state') == 'P':
            return last_body
        time.sleep(POLL_INTERVAL_SECONDS)
    raise AssertionError(f'Timed out waiting for published state. Last body={last_body}')

print('ADMIN_API_V1 =', ADMIN_API_V1)
print('MINTER_API_V1 =', MINTER_API_V1)
print('RESOLVER_API_V1 =', RESOLVER_API_V1)
print('STORE_API_BASE_URL =', STORE_API_BASE_URL)
print('IPFS_API_BASE_URL =', IPFS_API_BASE_URL)
print('IPFS_CLUSTER_API_URL =', IPFS_CLUSTER_API_URL)
print('AUTHORITY_ID =', AUTHORITY_ID)
print('NAAN =', NAAN)
print('TARGET_URL =', TARGET_URL)


## 1. Smoke Checks

The standalone worker, `dark-store-api`, and the IPFS cluster must already be running before continuing.


In [ ]:
admin_status = requests.get(f'{ADMIN_API_V1}/status', timeout=180)
show(admin_status)
assert admin_status.status_code == 200

minter_health = requests.get(f'{MINTER_BASE_URL}/health', timeout=180)
show(minter_health)
assert minter_health.status_code in {200, 503}

worker_status = requests.get(f'{MINTER_API_V1}/worker/status', headers=MINTER_HEADERS, timeout=180)
show(worker_status)
assert worker_status.status_code == 200
assert worker_status.json().get('running') is True, 'The worker must be running for this notebook.'

store_api_health = requests.get(f'{STORE_API_BASE_URL}/health', timeout=180)
show(store_api_health)
assert store_api_health.status_code == 200

ipfs_api_status = requests.post(f'{IPFS_API_BASE_URL}/api/v0/id', timeout=180)
show(ipfs_api_status)
assert ipfs_api_status.status_code == 200

ipfs_cluster_status = requests.get(f'{IPFS_CLUSTER_API_URL}/id', timeout=180)
show(ipfs_cluster_status)
assert ipfs_cluster_status.status_code == 200

resolver_health = requests.get(f'{RESOLVER_BASE_URL}/health', timeout=180)
show(resolver_health)
assert resolver_health.status_code == 200


## 2. Create the Authority and Then Assign the NAAN


In [ ]:
register_authority = requests.post(
    f'{ADMIN_API_V1}/authority',
    json={'uuid': AUTHORITY_ID, 'naans': [], 'fund_amount_eth': 0.05},
    timeout=180,
)
show(register_authority)
assert register_authority.status_code in {200, 201, 409}

get_authority = requests.get(f'{ADMIN_API_V1}/authority/{AUTHORITY_ID}', timeout=180)
show(get_authority)
assert get_authority.status_code == 200

authorize_naan = requests.post(
    f'{ADMIN_API_V1}/authority/{AUTHORITY_ID}/authorize-naan',
    json={'naan': NAAN},
    timeout=180,
)
show(authorize_naan)
assert authorize_naan.status_code == 200

get_naans = requests.get(f'{MINTER_API_V1}/authority/{AUTHORITY_ID}/naans', headers=MINTER_HEADERS, timeout=180)
show(get_naans)
assert get_naans.status_code == 200
assert NAAN in get_naans.json().get('naans', [])


## 3. Reserve an ARK


In [ ]:
reserve_ark = requests.post(
    f'{MINTER_API_V1}/arks',
    headers=MINTER_HEADERS,
    json={'authority_id': AUTHORITY_ID, 'naan': NAAN},
    timeout=180,
)
show(reserve_ark)
assert reserve_ark.status_code == 201
ARK = reserve_ark.json()['ark']
print('ARK =', ARK)


## 4. Stage Level-1 and Level-2 Metadata


In [ ]:
stage_metadata = requests.put(
    f'{MINTER_API_V1}/arks/{ARK}',
    headers=MINTER_HEADERS,
    json={
        'authority_id': AUTHORITY_ID,
        'target': TARGET_URL,
        'minimal_metadata': L1_METADATA,
        'original_metadata': L2_XML,
        'metadata_schema': 'oai_dc',
        'metadata_media_type': 'application/xml',
    },
    timeout=180,
)
show(stage_metadata)
assert stage_metadata.status_code == 200
assert stage_metadata.json()['state'] == 'D'
assert stage_metadata.json()['minimal_metadata']['title'] == L1_METADATA['title']


## 5. Wait Until the Worker Publishes the ARK


In [ ]:
published_ark = wait_until_published(ARK)
pprint(published_ark)
assert published_ark['state'] == 'P'
assert published_ark['target'] == TARGET_URL


## 6. Resolve the ARK to the Default Target


In [ ]:
resolve_default = requests.get(
    f'{RESOLVER_API_V1}/arks/{ARK}',
    allow_redirects=False,
    timeout=180,
)
show(resolve_default)
assert resolve_default.status_code in {302, 307}
assert resolve_default.headers['location'] == TARGET_URL


## 7. Resolve `?info` (Level-1 Projection Only)


In [ ]:
resolve_info = requests.get(f'{RESOLVER_API_V1}/arks/{ARK}?info', timeout=180)
show(resolve_info)
assert resolve_info.status_code == 200
assert resolve_info.json()['ark'] == ARK.replace('ark:', 'ark:/', 1)
assert resolve_info.json()['title'] == L1_METADATA['title']
assert 'original_metadata' not in resolve_info.json()
assert not any(key.endswith('cid') for key in resolve_info.json().keys())


## 8. Resolve `?metadata` (Original Level-2 Payload)


In [ ]:
resolve_metadata = requests.get(f'{RESOLVER_API_V1}/arks/{ARK}?metadata', timeout=180)
show(resolve_metadata)
assert resolve_metadata.status_code == 200
assert resolve_metadata.headers['content-type'].startswith('application/xml') or resolve_metadata.headers['content-type'].startswith('text/xml')
assert resolve_metadata.text.strip() == L2_XML

print('Notebook completed successfully for', ARK)


## 9. Verify That Level-1 and Level-2 Metadata Are Really Stored in IPFS

This is an operator-facing validation step. It uses the internal CIDs returned by the minter once the ARK reaches `PUBLISHED`, checks that `dark-store-api` reports them as pinned, and then reads the same content directly from the IPFS API.


In [ ]:
level1_cid = published_ark['level1_cid']
level2_cid = published_ark['level2_cid']

print('L1 CID =', level1_cid)
print('L2 CID =', level2_cid)

assert level1_cid
assert level2_cid

def wait_until_pinned(cid):
    deadline = time.time() + POLL_TIMEOUT_SECONDS
    last_body = None
    while time.time() < deadline:
        response = requests.get(f'{STORE_API_BASE_URL}/v1/status/{cid}', timeout=180)
        assert response.status_code == 200
        last_body = response.json()
        print('pin poll:', cid, last_body.get('status'), 'replicas=', last_body.get('replicas'))
        if last_body.get('pinned') is True:
            return last_body
        time.sleep(POLL_INTERVAL_SECONDS)
    raise AssertionError(f'Timed out waiting for pin status. cid={cid} last_body={last_body}')

l1_status = wait_until_pinned(level1_cid)
l2_status = wait_until_pinned(level2_cid)
pprint(l1_status)
pprint(l2_status)

l1_ipfs = requests.post(f'{IPFS_API_BASE_URL}/api/v0/cat', params={'arg': level1_cid}, timeout=180)
l2_ipfs = requests.post(f'{IPFS_API_BASE_URL}/api/v0/cat', params={'arg': level2_cid}, timeout=180)
show(l1_ipfs)
show(l2_ipfs)
assert l1_ipfs.status_code == 200
assert l2_ipfs.status_code == 200
assert l1_ipfs.json()['title'] == L1_METADATA['title']
assert l2_ipfs.text.strip() == L2_XML

print('IPFS verification completed successfully for', ARK)
